# PJM RTO Forecast Comparison — Load, Solar, Wind

Side-by-side comparison of all available DA forecasts for **RTO** on a given target date.

**Load forecasts**: PJM (official) vs GridStatus vs Meteologica
**Solar forecasts**: PJM (grid-scale + BTM) vs Meteologica
**Wind forecasts**: PJM vs Meteologica
**Net Load**: Load - Solar - Wind

## 1. Setup & Configuration

In [4]:
import sys
import logging
from pathlib import Path

In [5]:
import sys
import logging
from pathlib import Path

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ── Repo root & backend imports ──
REPO_ROOT = Path.cwd().parents[1]
SQL_DIR = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "backend"))

from src.utils.azure_postgresql import pull_from_db

# ── Logger ──
try:
    from src.utils.logging_utils import init_logging
    pl = init_logging(name="forecasts", log_to_file=False, level=logging.INFO)
    logger = pl.logger
except Exception:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
    logger = logging.getLogger("forecasts")

# ── Configurable Parameters ──
TARGET_DATE = pd.Timestamp.now().normalize()           # execution date (today)
FORECAST_DATE = TARGET_DATE + pd.Timedelta(days=1)     # forecast date (tomorrow)
HISTORY_DAYS = 3                                        # days of history for validation
MAX_EXECUTION_HOUR = 10                                 # DA cutoff hour

# Common SQL params
SQL_PARAMS = dict(
    execution_date=TARGET_DATE.date(),
    forecast_date=FORECAST_DATE.date(),
    max_execution_hour=MAX_EXECUTION_HOUR,
)

logger.info(f"Execution date: {TARGET_DATE.date()}")
logger.info(f"Forecast date:  {FORECAST_DATE.date()}")
logger.info(f"Max exec hour:  {MAX_EXECUTION_HOUR}")


def read_sql(filename: str, **extra_params) -> str:
    """Read a .sql file from the notebook directory and format with params."""
    params = {**SQL_PARAMS, **extra_params}
    return (SQL_DIR / filename).read_text().format(**params)

INFO:forecasts:Execution date: 2026-03-11
INFO:forecasts:Forecast date:  2026-03-12
INFO:forecasts:Max exec hour:  10


## 2. Data Pull — All Forecast Sources

In [6]:
# ── Load Forecasts ──
df_pjm_load = pull_from_db(query=read_sql("pjm_load_forecast.sql"))
df_gs_load = pull_from_db(query=read_sql("gridstatus_load_forecast.sql"))
df_met_load = pull_from_db(query=read_sql("meteologica_load_forecast.sql"))

# ── Solar Forecasts ──
df_pjm_solar = pull_from_db(query=read_sql("pjm_solar_forecast.sql"))
df_met_solar = pull_from_db(query=read_sql("meteologica_solar_forecast.sql"))

# ── Wind Forecasts ──
df_pjm_wind = pull_from_db(query=read_sql("pjm_wind_forecast.sql"))
df_met_wind = pull_from_db(query=read_sql("meteologica_wind_forecast.sql"))

# ── Summary ──
sources = {
    "PJM Load": df_pjm_load,
    "GridStatus Load": df_gs_load,
    "Meteologica Load": df_met_load,
    "PJM Solar": df_pjm_solar,
    "Meteologica Solar": df_met_solar,
    "PJM Wind": df_pjm_wind,
    "Meteologica Wind": df_met_wind,
}

for name, df in sources.items():
    if df is not None and len(df) > 0:
        logger.info(f"{name}: {len(df):,} rows")
    else:
        logger.warning(f"{name}: NO DATA")

INFO:forecasts:PJM Load: 24 rows
INFO:forecasts:GridStatus Load: 24 rows
INFO:forecasts:Meteologica Load: 24 rows
INFO:forecasts:PJM Solar: 24 rows
INFO:forecasts:Meteologica Solar: 24 rows
INFO:forecasts:PJM Wind: 24 rows
INFO:forecasts:Meteologica Wind: 24 rows


## 3. Data Validation — Previous 3 Days

Pull the previous 3 days of each forecast source to check completeness, nulls, outliers, and temporal continuity.

In [7]:
# Pull previous 3 days for each source (RTO)
history_dfs = {}
for days_back in range(1, HISTORY_DAYS + 1):
    hist_exec = (TARGET_DATE - pd.Timedelta(days=days_back)).date()
    hist_fcst = (FORECAST_DATE - pd.Timedelta(days=days_back)).date()
    p = dict(execution_date=hist_exec, forecast_date=hist_fcst, max_execution_hour=MAX_EXECUTION_HOUR)

    for name, sql_file, val_col in [
        ("PJM Load", "pjm_load_forecast.sql", "forecast_load_mw"),
        ("GridStatus Load", "gridstatus_load_forecast.sql", "forecast_load_mw"),
        ("Meteologica Load", "meteologica_load_forecast.sql", "forecast_load_mw"),
        ("PJM Solar", "pjm_solar_forecast.sql", "solar_forecast"),
        ("PJM Wind", "pjm_wind_forecast.sql", "wind_forecast"),
        ("Meteologica Solar", "meteologica_solar_forecast.sql", "forecast_generation_mw"),
        ("Meteologica Wind", "meteologica_wind_forecast.sql", "forecast_generation_mw"),
    ]:
        df = pull_from_db(query=read_sql(sql_file, **p))
        if df is not None and len(df) > 0:
            df["_source"] = name
            df["_val_col"] = val_col
            df["_hist_date"] = str(hist_fcst)
            history_dfs[f"{name}_{hist_fcst}"] = df

# ── Validation table ──
validation_rows = []
for key, df in history_dfs.items():
    name = df["_source"].iloc[0]
    val_col = df["_val_col"].iloc[0]
    hist_date = df["_hist_date"].iloc[0]
    n_hours = df["hour_ending"].nunique()
    nulls = df[val_col].isna().sum()
    vals = df[val_col].dropna()
    val_min = vals.min() if len(vals) > 0 else np.nan
    val_max = vals.max() if len(vals) > 0 else np.nan
    val_mean = vals.mean() if len(vals) > 0 else np.nan
    std = vals.std() if len(vals) > 1 else 0
    outliers = ((vals - val_mean).abs() > 3 * std).sum() if std > 0 else 0
    expected = set(range(1, 25))
    actual = set(df["hour_ending"].astype(int))
    missing = len(expected - actual)
    passed = n_hours == 24 and nulls == 0 and missing == 0

    validation_rows.append({
        "source": name, "forecast_date": hist_date, "hours": n_hours,
        "nulls": nulls, "missing_hours": missing,
        "min_mw": round(val_min, 0), "max_mw": round(val_max, 0),
        "mean_mw": round(val_mean, 0), "outliers": outliers, "pass": passed,
    })

df_val = pd.DataFrame(validation_rows)
total = len(df_val)
passed_count = df_val["pass"].sum()
failed_count = total - passed_count

display(df_val.style.set_caption("RTO Forecast Validation — Previous 3 Days").map(
    lambda v: "background-color: #2d5a2d" if v is True else ("background-color: #5a2d2d" if v is False else ""),
    subset=["pass"],
))
print(f"\nVALIDATION SUMMARY: {passed_count}/{total} passed, {failed_count}/{total} failed")

,source,forecast_date,hours,nulls,missing_hours,min_mw,max_mw,mean_mw,outliers,pass
0,PJM Load,2026-03-11,24,0,0,72024.000000,98253.000000,87157.000000,0,True
1,GridStatus Load,2026-03-11,24,0,0,0.000000,0.000000,0.000000,0,True
2,Meteologica Load,2026-03-11,24,0,0,72583.000000,96187.000000,86469.000000,0,True
3,PJM Solar,2026-03-11,24,0,0,0.000000,6090.000000,2069.000000,0,True
4,PJM Wind,2026-03-11,24,0,0,6039.000000,9449.000000,7853.000000,0,True
5,Meteologica Solar,2026-03-11,24,0,0,0.000000,5903.000000,1908.000000,0,True
6,Meteologica Wind,2026-03-11,24,0,0,6275.000000,8995.000000,7629.000000,0,True
7,PJM Load,2026-03-10,24,0,0,72930.000000,96525.000000,85542.000000,0,True
8,GridStatus Load,2026-03-10,24,0,0,72930.000000,96525.000000,85542.000000,0,True
9,Meteologica Load,2026-03-10,24,0,0,74625.000000,94934.000000,86090.000000,0,True



VALIDATION SUMMARY: 17/17 passed, 0/17 failed


## 4. Load Forecast Comparison — PJM vs GridStatus vs Meteologica

RTO region side-by-side comparison of all three load forecast sources.

In [10]:
# Build combined load DataFrame (all sources already filtered to RTO in SQL)
load_frames = []

for label, df in [("PJM", df_pjm_load), ("GridStatus", df_gs_load), ("Meteologica", df_met_load)]:
    if df is not None and len(df) > 0:
        tmp = df[["hour_ending", "forecast_load_mw"]].copy()
        tmp = tmp.rename(columns={"forecast_load_mw": "load_mw"})
        tmp["source"] = label

if load_frames:
    df_load_cmp = pd.concat(load_frames, ignore_index=True).sort_values(["source", "hour_ending"])

    # ── Line chart overlay ──
    fig = px.line(
        df_load_cmp, x="hour_ending", y="load_mw", color="source",
        title=f"RTO Load Forecast Comparison — {FORECAST_DATE.date()}",
        labels={"load_mw": "Load (MW)", "hour_ending": "Hour Ending", "source": "Source"},
        markers=True, template="plotly_dark",
    )
    fig.update_layout(height=500, xaxis=dict(dtick=1))
    fig.show()

    # ── Delta from PJM (reference) ──
    pjm_ref = df_load_cmp[df_load_cmp["source"] == "PJM"][["hour_ending", "load_mw"]].set_index("hour_ending")

    delta_frames = []
    for label in ["GridStatus", "Meteologica"]:
        src = df_load_cmp[df_load_cmp["source"] == label][["hour_ending", "load_mw"]].set_index("hour_ending")
        delta = (src["load_mw"] - pjm_ref["load_mw"]).reset_index()
        delta.columns = ["hour_ending", "delta_mw"]
        delta["source"] = f"{label} - PJM"
        delta_frames.append(delta)

    if delta_frames:
        df_load_delta = pd.concat(delta_frames, ignore_index=True)
        fig = px.bar(
            df_load_delta, x="hour_ending", y="delta_mw", color="source", barmode="group",
            title=f"RTO Load Forecast Delta vs PJM — {FORECAST_DATE.date()}",
            labels={"delta_mw": "Delta (MW)", "hour_ending": "Hour Ending"},
            template="plotly_dark",
        )
        fig.add_hline(y=0, line_color="gray", line_width=0.5)
        fig.update_layout(height=400, xaxis=dict(dtick=1))
        fig.show()
else:
    print("No load forecast data available.")

No load forecast data available.


In [11]:
load_frames

[]

## 5. Solar Forecast Comparison — PJM vs Meteologica

PJM provides grid-scale (`solar_forecast`) and behind-the-meter (`solar_forecast_btm`).
Meteologica provides a single RTO solar generation forecast.

In [ ]:
solar_frames = []

if df_pjm_solar is not None and len(df_pjm_solar) > 0:
    tmp = df_pjm_solar[["hour_ending", "solar_forecast"]].copy()
    tmp = tmp.rename(columns={"solar_forecast": "solar_mw"})
    tmp["source"] = "PJM Grid-Scale"
    solar_frames.append(tmp)

    tmp_btm = df_pjm_solar[["hour_ending", "solar_forecast_btm"]].copy()
    tmp_btm = tmp_btm.rename(columns={"solar_forecast_btm": "solar_mw"})
    tmp_btm["source"] = "PJM BTM"
    solar_frames.append(tmp_btm)

    tmp_total = df_pjm_solar[["hour_ending"]].copy()
    tmp_total["solar_mw"] = df_pjm_solar["solar_forecast"] + df_pjm_solar["solar_forecast_btm"]
    tmp_total["source"] = "PJM Total"
    solar_frames.append(tmp_total)

if df_met_solar is not None and len(df_met_solar) > 0:
    tmp = df_met_solar[["hour_ending", "forecast_generation_mw"]].copy()
    tmp = tmp.rename(columns={"forecast_generation_mw": "solar_mw"})
    tmp["source"] = "Meteologica"
    solar_frames.append(tmp)

if solar_frames:
    df_solar_cmp = pd.concat(solar_frames, ignore_index=True).sort_values(["source", "hour_ending"])

    fig = px.line(
        df_solar_cmp, x="hour_ending", y="solar_mw", color="source",
        title=f"RTO Solar Forecast Comparison — {FORECAST_DATE.date()}",
        labels={"solar_mw": "Solar (MW)", "hour_ending": "Hour Ending"},
        markers=True, template="plotly_dark",
    )
    fig.update_layout(height=500, xaxis=dict(dtick=1))
    fig.show()

    # ── Stacked area: PJM grid-scale + BTM ──
    if df_pjm_solar is not None and len(df_pjm_solar) > 0:
        df_stack = df_pjm_solar[["hour_ending", "solar_forecast", "solar_forecast_btm"]].copy().sort_values("hour_ending")
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=df_stack["hour_ending"], y=df_stack["solar_forecast"],
            name="Grid-Scale", stackgroup="one", line=dict(color="#FFA15A"),
        ))
        fig.add_trace(go.Scatter(
            x=df_stack["hour_ending"], y=df_stack["solar_forecast_btm"],
            name="Behind-the-Meter", stackgroup="one", line=dict(color="#FECB52"),
        ))
        fig.update_layout(
            title=f"RTO PJM Solar Forecast Breakdown — {FORECAST_DATE.date()}",
            xaxis_title="Hour Ending", yaxis_title="Solar (MW)",
            template="plotly_dark", height=400, xaxis=dict(dtick=1),
        )
        fig.show()
else:
    print("No solar forecast data available.")

## 6. Wind Forecast Comparison — PJM vs Meteologica

In [ ]:
wind_frames = []

if df_pjm_wind is not None and len(df_pjm_wind) > 0:
    tmp = df_pjm_wind[["hour_ending", "wind_forecast"]].copy()
    tmp = tmp.rename(columns={"wind_forecast": "wind_mw"})
    tmp["source"] = "PJM"
    wind_frames.append(tmp)

if df_met_wind is not None and len(df_met_wind) > 0:
    tmp = df_met_wind[["hour_ending", "forecast_generation_mw"]].copy()
    tmp = tmp.rename(columns={"forecast_generation_mw": "wind_mw"})
    tmp["source"] = "Meteologica"
    wind_frames.append(tmp)

if wind_frames:
    df_wind_cmp = pd.concat(wind_frames, ignore_index=True).sort_values(["source", "hour_ending"])

    fig = px.line(
        df_wind_cmp, x="hour_ending", y="wind_mw", color="source",
        title=f"RTO Wind Forecast Comparison — {FORECAST_DATE.date()}",
        labels={"wind_mw": "Wind (MW)", "hour_ending": "Hour Ending"},
        markers=True, template="plotly_dark",
    )
    fig.update_layout(height=500, xaxis=dict(dtick=1))
    fig.show()

    # ── Delta: Meteologica - PJM ──
    if len(wind_frames) == 2:
        pjm_w = df_wind_cmp[df_wind_cmp["source"] == "PJM"].set_index("hour_ending")["wind_mw"]
        met_w = df_wind_cmp[df_wind_cmp["source"] == "Meteologica"].set_index("hour_ending")["wind_mw"]
        delta = (met_w - pjm_w).reset_index()
        delta.columns = ["hour_ending", "delta_mw"]

        colors = ["#2ca02c" if v >= 0 else "#d62728" for v in delta["delta_mw"]]
        fig = go.Figure(go.Bar(
            x=delta["hour_ending"], y=delta["delta_mw"],
            marker_color=colors,
            text=[f"{v:+,.0f}" for v in delta["delta_mw"]],
            textposition="outside", textfont=dict(size=9),
            hovertemplate="HE%{x}<br>Delta: %{y:+,.0f} MW<extra></extra>",
        ))
        fig.add_hline(y=0, line_color="gray", line_width=0.5)
        fig.update_layout(
            title=f"RTO Wind Forecast Delta (Meteologica - PJM) — {FORECAST_DATE.date()}",
            xaxis_title="Hour Ending", yaxis_title="Delta (MW)",
            template="plotly_dark", height=400, xaxis=dict(dtick=1),
        )
        fig.show()
else:
    print("No wind forecast data available.")

## 7. Net Load — Load minus Solar minus Wind

Net load = Load - Solar (total) - Wind, using PJM forecasts.
Shows the residual demand that must be met by dispatchable generation.

In [ ]:
# Build net load from PJM sources (already RTO from SQL)
has_load = df_pjm_load is not None and len(df_pjm_load) > 0
has_solar = df_pjm_solar is not None and len(df_pjm_solar) > 0
has_wind = df_pjm_wind is not None and len(df_pjm_wind) > 0

if has_load and has_solar and has_wind:
    df_net = df_pjm_load[["hour_ending", "forecast_load_mw"]].copy()
    df_net = df_net.sort_values("hour_ending").reset_index(drop=True)
    df_net = df_net.rename(columns={"forecast_load_mw": "load_mw"})

    solar = df_pjm_solar[["hour_ending", "solar_forecast", "solar_forecast_btm"]].sort_values("hour_ending").reset_index(drop=True)
    df_net["solar_total_mw"] = (solar["solar_forecast"] + solar["solar_forecast_btm"]).values

    wind = df_pjm_wind[["hour_ending", "wind_forecast"]].sort_values("hour_ending").reset_index(drop=True)
    df_net["wind_mw"] = wind["wind_forecast"].values

    df_net["net_load_mw"] = df_net["load_mw"] - df_net["solar_total_mw"] - df_net["wind_mw"]

    # ── Stacked area: Net Load decomposition ──
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_net["hour_ending"], y=df_net["net_load_mw"],
        name="Net Load (dispatchable)", fill="tozeroy",
        line=dict(color="#636EFA", width=2),
    ))
    fig.add_trace(go.Scatter(
        x=df_net["hour_ending"], y=df_net["net_load_mw"] + df_net["wind_mw"],
        name="+ Wind", fill="tonexty",
        line=dict(color="#00CC96", width=1),
    ))
    fig.add_trace(go.Scatter(
        x=df_net["hour_ending"], y=df_net["net_load_mw"] + df_net["wind_mw"] + df_net["solar_total_mw"],
        name="+ Solar (= Gross Load)", fill="tonexty",
        line=dict(color="#FFA15A", width=1),
    ))
    fig.update_layout(
        title=f"RTO Net Load Decomposition — {FORECAST_DATE.date()}",
        xaxis_title="Hour Ending", yaxis_title="MW",
        template="plotly_dark", height=500, xaxis=dict(dtick=1),
    )
    fig.show()

    # ── Line overlay: Gross vs Net ──
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df_net["hour_ending"], y=df_net["load_mw"],
        name="Gross Load", line=dict(color="#EF553B", width=2.5), mode="lines+markers",
    ))
    fig.add_trace(go.Scatter(
        x=df_net["hour_ending"], y=df_net["net_load_mw"],
        name="Net Load", line=dict(color="#636EFA", width=2.5), mode="lines+markers",
    ))
    fig.update_layout(
        title=f"RTO Gross Load vs Net Load — {FORECAST_DATE.date()}",
        xaxis_title="Hour Ending", yaxis_title="MW",
        template="plotly_dark", height=450, xaxis=dict(dtick=1),
    )
    fig.show()

    print(f"Peak Gross Load:  {df_net['load_mw'].max():>10,.0f} MW  (HE{df_net.loc[df_net['load_mw'].idxmax(), 'hour_ending']:.0f})")
    print(f"Peak Net Load:    {df_net['net_load_mw'].max():>10,.0f} MW  (HE{df_net.loc[df_net['net_load_mw'].idxmax(), 'hour_ending']:.0f})")
    print(f"Min Net Load:     {df_net['net_load_mw'].min():>10,.0f} MW  (HE{df_net.loc[df_net['net_load_mw'].idxmin(), 'hour_ending']:.0f})")
    print(f"Peak Solar:       {df_net['solar_total_mw'].max():>10,.0f} MW  (HE{df_net.loc[df_net['solar_total_mw'].idxmax(), 'hour_ending']:.0f})")
    print(f"Peak Wind:        {df_net['wind_mw'].max():>10,.0f} MW  (HE{df_net.loc[df_net['wind_mw'].idxmax(), 'hour_ending']:.0f})")
else:
    missing = [n for n, ok in [("Load", has_load), ("Solar", has_solar), ("Wind", has_wind)] if not ok]
    print(f"Cannot compute net load — missing: {', '.join(missing)}")

## 8. RTO Forecast Summary Table

All forecasts at a glance: peak, min, mean, and peak hour for each source.

In [ ]:
summary_rows = []

# Load sources (already RTO from SQL)
for label, df, col in [
    ("PJM Load", df_pjm_load, "forecast_load_mw"),
    ("GridStatus Load", df_gs_load, "forecast_load_mw"),
    ("Meteologica Load", df_met_load, "forecast_load_mw"),
]:
    if df is not None and len(df) > 0:
        src = df.sort_values("hour_ending")
        summary_rows.append({
            "Source": label, "Type": "Load",
            "Peak (MW)": f"{src[col].max():,.0f}",
            "Peak HE": int(src.loc[src[col].idxmax(), "hour_ending"]),
            "Min (MW)": f"{src[col].min():,.0f}",
            "Min HE": int(src.loc[src[col].idxmin(), "hour_ending"]),
            "Mean (MW)": f"{src[col].mean():,.0f}",
        })

# Solar sources
if df_pjm_solar is not None and len(df_pjm_solar) > 0:
    vals = df_pjm_solar["solar_forecast"] + df_pjm_solar["solar_forecast_btm"]
    he = df_pjm_solar["hour_ending"]
    summary_rows.append({
        "Source": "PJM Solar (total)", "Type": "Solar",
        "Peak (MW)": f"{vals.max():,.0f}",
        "Peak HE": int(he.iloc[vals.idxmax()]),
        "Min (MW)": f"{vals.min():,.0f}",
        "Min HE": int(he.iloc[vals.idxmin()]),
        "Mean (MW)": f"{vals.mean():,.0f}",
    })

if df_met_solar is not None and len(df_met_solar) > 0:
    src = df_met_solar.sort_values("hour_ending")
    summary_rows.append({
        "Source": "Meteologica Solar", "Type": "Solar",
        "Peak (MW)": f"{src['forecast_generation_mw'].max():,.0f}",
        "Peak HE": int(src.loc[src["forecast_generation_mw"].idxmax(), "hour_ending"]),
        "Min (MW)": f"{src['forecast_generation_mw'].min():,.0f}",
        "Min HE": int(src.loc[src["forecast_generation_mw"].idxmin(), "hour_ending"]),
        "Mean (MW)": f"{src['forecast_generation_mw'].mean():,.0f}",
    })

# Wind sources
for label, df, col in [
    ("PJM Wind", df_pjm_wind, "wind_forecast"),
    ("Meteologica Wind", df_met_wind, "forecast_generation_mw"),
]:
    if df is not None and len(df) > 0:
        src = df.sort_values("hour_ending")
        summary_rows.append({
            "Source": label, "Type": "Wind",
            "Peak (MW)": f"{src[col].max():,.0f}",
            "Peak HE": int(src.loc[src[col].idxmax(), "hour_ending"]),
            "Min (MW)": f"{src[col].min():,.0f}",
            "Min HE": int(src.loc[src[col].idxmin(), "hour_ending"]),
            "Mean (MW)": f"{src[col].mean():,.0f}",
        })

# Net load
if has_load and has_solar and has_wind:
    summary_rows.append({
        "Source": "PJM Net Load", "Type": "Net Load",
        "Peak (MW)": f"{df_net['net_load_mw'].max():,.0f}",
        "Peak HE": int(df_net.loc[df_net["net_load_mw"].idxmax(), "hour_ending"]),
        "Min (MW)": f"{df_net['net_load_mw'].min():,.0f}",
        "Min HE": int(df_net.loc[df_net["net_load_mw"].idxmin(), "hour_ending"]),
        "Mean (MW)": f"{df_net['net_load_mw'].mean():,.0f}",
    })

df_summary = pd.DataFrame(summary_rows)

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=list(df_summary.columns),
        fill_color="#1f2937",
        font=dict(color="white", size=12),
        align="center",
    ),
    cells=dict(
        values=[df_summary[col] for col in df_summary.columns],
        fill_color=[["#111827"] * len(df_summary)],
        font=dict(color="white", size=11),
        align="center",
    ),
)])
fig_table.update_layout(
    title=f"RTO Forecast Summary — {FORECAST_DATE.date()}",
    template="plotly_dark",
    height=60 + 30 * len(df_summary),
    margin=dict(l=20, r=20, t=50, b=20),
)
fig_table.show()

df_summary